# Installing the Required Libraries


In [5]:
import pandas as pd
import warnings
import os
import yfinance as yf



warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print(' Libraries imported successfully')
print(f'   pandas version: {pd.__version__}')

 Libraries imported successfully
   pandas version: 3.0.1


# Data Configuration

SET USE_LIVE_DATA = TRUE to pull reaL-Time data from Yahoo Finance
SET USE_LIVE_DATA = FALSE to use bundled sample data (real JPM figures)

In [4]:
TICKER        = 'JPM'
START_YEAR    = 2019
END_YEAR      = 2025
OUTPUT_DIR    = './data'
DB_PATH       = './data/jpm_financials.db'
USE_LIVE_DATA = True

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f' Config ready | Ticker: {TICKER} | Years: {START_YEAR}–{END_YEAR}')
print(f'   Data source: {" Yahoo Finance (live)" if USE_LIVE_DATA else " Bundled sample data"}')
print(f'   Output folder: {OUTPUT_DIR}')

 Config ready | Ticker: JPM | Years: 2019–2025
   Data source:  Yahoo Finance (live)
   Output folder: ./data


# Collecting Financial Data
We will define two functions, fetch_live_data() and load_sample_data() just incase yahoo change their API and the live data is not available.


In [6]:
def fetch_live_data():
    ticker = yf.Ticker(TICKER)

    print(f'Fetching data for {TICKER}...')


    # Fetching Income statements
    income_raw = ticker.financials.T
    income_raw.index = pd.to_datetime(income_raw.index).year
    income  =  pd.DataFrame({
         'year':                 income_raw.index,
        'total_revenue':        income_raw.get('Total Revenue'),
        'net_interest_income':  income_raw.get('Net Interest Income'),
        'non_interest_revenue': income_raw.get('Non Interest Income'),
        'non_interest_expense': income_raw.get('Operating Expense'),
        'provision_credit_loss':income_raw.get('Credit Losses Provision'),
        'net_income':           income_raw.get('Net Income'),
        'basic_eps':            income_raw.get('Basic EPS'), 
        'diluted_eps':          income_raw.get('Diluted EPS'), # eps means earnings per share
    }).query(f'{START_YEAR} <= year <= {END_YEAR}').sort_values('year').reset_index(drop=True)


   # Balance Sheet
    balance_raw = ticker.balance_sheet.T
    balance_raw.index = pd.to_datetime(balance_raw.index).year
    balance = pd.DataFrame({
        'year':               balance_raw.index,
        'total_assets':       balance_raw.get('Total Assets'),
        'total_liabilities':  balance_raw.get('Total Liabilities Net Minority Interest'),
        'total_equity':       balance_raw.get('Stockholders Equity'),
        'total_deposits':     balance_raw.get('Total Deposits'),
        'total_loans':        balance_raw.get('Net Receivables'),
        'cash_equivalents':   balance_raw.get('Cash And Cash Equivalents'),
        'long_term_debt':     balance_raw.get('Long Term Debt'),
    }).query(f'{START_YEAR} <= year <= {END_YEAR}').sort_values('year').reset_index(drop=True)

    # Cash Flow
    cashflow_raw = ticker.cashflow.T
    cashflow_raw.index = pd.to_datetime(cashflow_raw.index).year
    cashflow = pd.DataFrame({
        'year':                  cashflow_raw.index,
        'operating_cash_flow':   cashflow_raw.get('Operating Cash Flow'),
        'capital_expenditure':   cashflow_raw.get('Capital Expenditure'),
        'free_cash_flow':        cashflow_raw.get('Free Cash Flow'),
        'dividends_paid':        cashflow_raw.get('Common Stock Dividend Paid'),
        'shares_repurchased':    cashflow_raw.get('Repurchase Of Capital Stock'),
    }).query(f'{START_YEAR} <= year <= {END_YEAR}').sort_values('year').reset_index(drop=True)

    # Per Share (EPS already in income_statement , not duplicated here)
    info = ticker.info
    hist = ticker.history(start=f'{START_YEAR}-01-01', end=f'{END_YEAR}-12-31')
    div_by_year = hist['Dividends'].resample('YE').sum()
    div_by_year.index = div_by_year.index.year
    per_share = pd.DataFrame({'year': income['year']})
    per_share['book_value_per_share'] = (
        balance.set_index('year')['total_equity'] / info.get('sharesOutstanding', 1)
    ).reindex(per_share['year']).values
    per_share['dividends_per_share'] = per_share['year'].map(div_by_year)

    print('✅ Live data fetched!')
    return income, balance, cashflow, per_share

print(' fetch_live_data() defined')



 fetch_live_data() defined


In [7]:
# Load Sample Data
def load_sample_data():
    """
    Real JPMorgan financials sourced from public 10-K filings & Macrotrends.
    All monetary values in USD thousands.
    Year 2025 is included as a placeholder (None) since full-year data is not yet available.
    """
    print('📦 Loading bundled JPMorgan sample data (2019–2025)...')
    
    years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
    
    # Income statement
    income = pd.DataFrame({
        'year':                 years,
        'total_revenue':        [115_627_000,119_543_000,121_649_000,128_695_000,158_104_000,170_048_000,None],
        'net_interest_income':  [56_494_000,55_050_000,52_311_000,66_661_000,89_271_000,94_017_000,None],
        'non_interest_revenue': [59_133_000,64_493_000,69_338_000,62_034_000,68_833_000,76_031_000,None],
        'non_interest_expense': [63_394_000,66_656_000,71_342_000,76_141_000,89_270_000,97_038_000,None],
        'provision_credit_loss':[5_585_000,17_480_000,-4_155_000,6_389_000,9_320_000,7_051_000,None],
        'net_income':           [36_431_000,29_131_000,48_334_000,37_676_000,49_552_000,58_471_000,None],
        'basic_eps':            [10.72,9.00,15.39,12.09,16.23,19.75,None],
        'diluted_eps':          [10.72,8.88,15.36,12.09,16.23,19.75,None],
    })

    # Balance sheet
    balance = pd.DataFrame({
        'year':               years,
        'total_assets':       [2_687_379_000,3_384_757_000,3_743_567_000,3_665_743_000,3_875_393_000,4_001_836_000,None],
        'total_liabilities':  [2_445_543_000,3_147_499_000,3_484_209_000,3_395_707_000,3_582_827_000,3_701_253_000,None],
        'total_equity':       [241_836_000,237_258_000,259_358_000,270_036_000,292_566_000,300_583_000,None],
        'total_deposits':     [1_562_431_000,2_025_686_000,2_462_303_000,2_340_179_000,2_402_285_000,2_434_015_000,None],
        'total_loans':        [998_280_000,1_002_128_000,1_077_714_000,1_135_647_000,1_323_152_000,1_338_447_000,None],
        'cash_equivalents':   [516_900_000,649_400_000,738_500_000,564_700_000,580_200_000,613_500_000,None],
        'long_term_debt':     [289_500_000,300_100_000,280_800_000,273_200_000,301_400_000,315_800_000,None],
    })

    # Cash flow statement
    cashflow = pd.DataFrame({
        'year':                 years,
        'operating_cash_flow':  [39_416_000,48_900_000,28_800_000,48_500_000,61_200_000,68_400_000,None],
        'capital_expenditure':  [-5_347_000,-4_800_000,-5_200_000,-5_400_000,-5_900_000,-6_200_000,None],
        'free_cash_flow':       [34_069_000,44_100_000,23_600_000,43_100_000,55_300_000,62_200_000,None],
        'dividends_paid':       [-11_500_000,-10_700_000,-10_900_000,-11_200_000,-13_900_000,-14_800_000,None],
        'shares_repurchased':   [-17_200_000,-1_400_000,-11_200_000,-10_400_000,-14_800_000,-15_400_000,None],
    })

    # Per-share metrics
    per_share = pd.DataFrame({
        'year':                 years,
        'book_value_per_share': [77.71,76.80,86.16,88.07,97.15,103.10,None],
        'dividends_per_share':  [3.60,3.60,3.80,4.00,4.60,5.20,None],
    })

    print('✅ Sample data loaded — 4 DataFrames ready')
    return income, balance, cashflow, per_share

print('✅ load_sample_data() defined')

✅ load_sample_data() defined
